In [1]:
!pip install  cryptography


In [3]:
# Import necessary modules
import cryptography
import json
import os
import base64
import getpass
import secrets
import string

from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes


In [4]:
# File to store encrypted password entries
DB_FILE = "passwords.json"

In [8]:
def get_key_from_password(password: str, salt: bytes) -> bytes:
    # Use PBKDF2 key derivation function with SHA256
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),   # Hashing algorithm
        length=32,                   # Length of the output key
        salt=salt,                   # Unique salt
        iterations=100_000,          # Number of iterations for security
        backend=default_backend()
    )

    # Derive the key and return it in a URL-safe base64 format
    key = base64.urlsafe_b64encode(kdf.derive(password.encode()))
    return key

In [9]:
salt = os.urandom(16)  # Generate a new random salt
password = "my_secret_password"
key = get_key_from_password(password, salt)

b'M0oR8iObtu_YK8shBjGzb-y21CIC1foEorehStICnPM='


In [10]:
# Load existing database or create a new one with a fresh salt
def load_database():
    if not os.path.exists(DB_FILE):
        # If no file exists, create a new one with a random salt
        salt = os.urandom(16)  # Generate a random 16-byte salt
        with open(DB_FILE, "w") as f:
            json.dump({"salt": base64.b64encode(salt).decode(), "entries": []}, f)
        return salt, []
    else:
        # Load the salt and encrypted entries from the file
        with open(DB_FILE, "r") as f:
            data = json.load(f)
            salt = base64.b64decode(data["salt"])
            return salt, data["entries"]

In [11]:
# Save the salt and encrypted entries to the database file
def save_database(salt, entries):
    with open(DB_FILE, "w") as f:
        json.dump({
            "salt": base64.b64encode(salt).decode(),  # Save salt in base64 format
            "entries": entries
        }, f, indent=2)

In [12]:
# Encrypt a single password entry (a dictionary of info)
def encrypt_entry(entry, fernet):
    # Convert the entry to a JSON string, then encrypt it
    json_entry = json.dumps(entry)
    encrypted = fernet.encrypt(json_entry.encode())
    return base64.b64encode(encrypted).decode()  # Store as base64 for JSON compatibility

In [13]:
# Decrypt a password entry
def decrypt_entry(encrypted_entry, fernet):
    try:
        encrypted = base64.b64decode(encrypted_entry)
        decrypted = fernet.decrypt(encrypted)
        return json.loads(decrypted.decode())  # Convert JSON back to a dictionary
    except Exception:
        # If decryption fails (e.g., wrong master password), return None
        return None


In [14]:
# Generate a random password with letters, digits, and symbols
def generate_password(length):
    chars = string.ascii_letters + string.digits + string.punctuation
    return ''.join(secrets.choice(chars) for _ in range(length))


In [15]:
# Main function for the CLI interface
def main():
    print("🔐 Welcome to the Simple Password Manager")

    # Ask user for the master password (input hidden)
    master_password = getpass.getpass("Enter master password: ")

    # Load or initialize database and get encryption key
    salt, encrypted_entries = load_database()
    key = get_key_from_password(master_password, salt)  # Key is already base64-encoded
    fernet = Fernet(key)

    while True:
        # Display the menu
        print("\nChoose an option:")
        print("1. Save a new password")
        print("2. View saved passwords")
        print("3. Generate a random password")
        print("4. Exit")

        choice = input("Enter your choice: ").strip()

        if choice == "1":
            # Save a new password entry
            title = input("Title: ")
            username = input("Username/Email: ")
            password = getpass.getpass("Password: ")
            url = input("Website URL: ")
            note = input("Optional note: ")

            entry = {
                "title": title,
                "username": username,
                "password": password,
                "url": url,
                "note": note
            }

            encrypted = encrypt_entry(entry, fernet)
            encrypted_entries.append(encrypted)
            save_database(salt, encrypted_entries)
            print("✅ Password saved successfully!")

        elif choice == "2":
            print("\n📋 Saved Passwords:")
            for idx, enc in enumerate(encrypted_entries):
                entry = decrypt_entry(enc, fernet)
                if entry:
                    print(f"\n[{idx + 1}] {entry['title']}")
                    print(f"    Username: {entry['username']}")
                    print(f"    Password: {entry['password']}")
                    print(f"    URL: {entry['url']}")
                    print(f"    Note: {entry['note']}")
                else:
                    print(f"\n[{idx + 1}] 🔐 Could not decrypt (wrong master password?)")

        elif choice == "3":
            length = input("Enter length (16/24/32): ").strip()
            if length in ["16", "24", "32"]:
                pw = generate_password(int(length))
                print(f"🔑 Generated password: {pw}")
            else:
                print("❌ Invalid length. Please choose 16, 24, or 32.")

        elif choice == "4":
            print("👋 Goodbye!")
            break
        else:
            print("❌ Invalid choice. Try again.")


In [16]:
# Run the program
if __name__ == "__main__":
    main()


🔐 Welcome to the Simple Password Manager
Enter master password: ··········

Choose an option:
1. Save a new password
2. View saved passwords
3. Generate a random password
4. Exit
Enter your choice: 2

📋 Saved Passwords:

Choose an option:
1. Save a new password
2. View saved passwords
3. Generate a random password
4. Exit
Enter your choice: 1
Title: Srinaath
Username/Email: davidanbu1108@gmail.com
Password: ··········
Website URL: none
Optional note: no website
✅ Password saved successfully!

Choose an option:
1. Save a new password
2. View saved passwords
3. Generate a random password
4. Exit
Enter your choice: 2

📋 Saved Passwords:

[1] Srinaath
    Username: davidanbu1108@gmail.com
    Password: srinaath
    URL: none
    Note: no website

Choose an option:
1. Save a new password
2. View saved passwords
3. Generate a random password
4. Exit
Enter your choice: 4
👋 Goodbye!
